In [8]:
!pip install geobr
import pandas as pd
import numpy as np
import warnings
from google.colab import drive
import geobr
import geopandas
import matplotlib.pyplot as plt
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
dados = pd.read_excel('/content/drive/My Drive/covid/covid.xlsx')
dados.drop(['regiao', 'codRegiaoSaude','nomeRegiaoSaude', 'semanaEpi','Recuperadosnovos', 'emAcompanhamentoNovos', 'populacaoTCU2019', 'interior/metropolitana'],axis=1,inplace=True)
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641652 entries, 0 to 641651
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   estado           641505 non-null  object        
 1   municipio        635100 non-null  object        
 2   coduf            641652 non-null  int64         
 3   codmun           637536 non-null  float64       
 4   data             641652 non-null  datetime64[ns]
 5   casosAcumulado   641652 non-null  int64         
 6   casosNovos       641652 non-null  int64         
 7   obitosAcumulado  641652 non-null  int64         
 8   obitosNovos      641652 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(5), object(2)
memory usage: 44.1+ MB


In [10]:
def clipar(x):
  if x == np.inf or x == -np.inf:
    return None
  return x

def clipar2(x):
  if x > 5:
    return None
  return x

janela = 7
janela2 = 14
janelaC = 5

dados['mediamovelcaso'] = 0
dados['mediamovelobito'] = 0
dados['crescimentoobito'] = 0.0
dados['crescimentocasos'] = 0

merge1 = dados[~dados.codmun.notnull() & ~dados.municipio.notnull()].copy()
merge1.sort_values(by=['data'],inplace=True)

estados = pd.DataFrame()
for c in merge1['coduf'].unique():

  estado = merge1[merge1.coduf==c].copy()
  estado['mediamovelcaso'] = estado.iloc[:,6].rolling(window=janela).mean()
  estado['mediamovelobito'] = estado.iloc[:,8].rolling(window=janela).mean()
  estado['crescimentocasos'] = np.clip(np.log10(estado['casosAcumulado']),0,6)

  estado['crescimentoobito'] = (estado.iloc[:,10]-estado.iloc[:,10].shift(janela2))/estado.iloc[:,10].shift(janela2)
  #estado['crescimentoobito'] = np.clip(estado['crescimentoobito'],-janelaC,janelaC)
  estado['crescimentoobito'].fillna(method='ffill', inplace=True)

  estados = pd.concat([estados, estado])

nacional = estados[estados.coduf==76]
estados = estados[estados.coduf!=76]
nacional.drop(['estado','municipio','coduf','codmun'],axis=1,inplace=True)
nacional.info()
nacional.to_csv('/content/drive/My Drive/covid/dados/nacional.csv', index = False, header=True)

estados.info()
estados.to_csv('/content/drive/My Drive/covid/dados/estadual.csv', index = False, header=True)

evolucao = estados.drop(['municipio', 'coduf', 'codmun', 'casosAcumulado', 'casosNovos', 'obitosAcumulado', 'obitosNovos', 'mediamovelcaso', 'mediamovelobito','crescimentocasos' ],axis=1)
evolucao = pd.pivot(evolucao,index='data',columns='estado',values='crescimentoobito')
for c in evolucao.columns:
  evolucao[c] = evolucao[c].apply(clipar)
evolucao = evolucao.transpose()
for c in evolucao.columns:
  evolucao[c] = evolucao[c].rank(method='min',ascending=False)
  evolucao[c] = evolucao[c].apply(clipar2)
evolucao = evolucao.transpose()
evolucao.info()
evolucao.tail(janela2).dropna(how='all', axis=1).to_csv('/content/drive/My Drive/covid/dados/evolucao.csv', header=True)

merge2 = dados[dados.codmun.notnull() & dados.municipio.notnull()].copy()
merge2.sort_values(by=['data'],inplace=True)

municipios = pd.DataFrame()
for c in merge2['codmun'].unique():

  municipio = merge2[merge2.codmun==c].copy()
  municipio['mediamovelcaso'] = municipio.iloc[:,6].rolling(window=janela).mean()
  municipio['mediamovelobito'] = municipio.iloc[:,8].rolling(window=janela).mean()
  municipio['crescimentocasos'] = np.clip(np.log10(municipio['casosAcumulado']),0,6)

  municipio['crescimentoobito'] = (municipio.iloc[:,10]-municipio.iloc[:,10].shift(janela2))/municipio.iloc[:,10].shift(janela2)
  municipio['crescimentoobito'] = np.clip(municipio['crescimentoobito'],-janelaC,janelaC)
  municipio['crescimentoobito'].fillna(method='ffill', inplace=True)

  municipios = pd.concat([municipios, municipio])

for e in estados['estado'].unique():
  municipios[municipios.estado==e].to_csv('/content/drive/My Drive/covid/dados/'+e+'.csv', index = False, header=True)
municipios.info()

del dados
del municipio
del estado
del nacional
del evolucao
del merge2
del merge1

/usr/local/lib/python3.6/dist-packages/pandas/core/series.py:679: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


<class 'pandas.core.frame.DataFrame'>
Int64Index: 147 entries, 0 to 146
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   data              147 non-null    datetime64[ns]
 1   casosAcumulado    147 non-null    int64         
 2   casosNovos        147 non-null    int64         
 3   obitosAcumulado   147 non-null    int64         
 4   obitosNovos       147 non-null    int64         
 5   mediamovelcaso    141 non-null    float64       
 6   mediamovelobito   141 non-null    float64       
 7   crescimentoobito  126 non-null    float64       
 8   crescimentocasos  147 non-null    float64       
dtypes: datetime64[ns](1), float64(4), int64(4)
memory usage: 11.5 KB
<class 'pandas.core.frame.DataFrame'>
Int64Index: 3969 entries, 882 to 1175
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   estado    

In [17]:
import imageio
from os import listdir

def criagif(nome):
  filenames = ['/content/drive/My Drive/covid/img/'+nome+'/'+a for a in listdir('/content/drive/My Drive/covid/img/'+nome+'/')]
  with imageio.get_writer('/content/drive/My Drive/covid/'+nome+'.gif', mode='I') as writer:
      for filename in filenames:
          image = imageio.imread(filename)
          writer.append_data(image)

def cor(valor):
  if valor>=0.15:
    return 4
  elif valor<=-0.15:
    return 2
  elif abs(valor)<0.15:
    return 3
  else:
    return 1

def plotting(v,i,color,ax):
  if len(v[v['cor']==i])>0:
    v[v['cor']==i].plot(color=color,ax=ax)

def corta(i):
  return int(i) // 10

%matplotlib inline

def plota_nacional_mun():

  prontas = listdir('/content/drive/My Drive/covid/img/nacional_mun/')

  for d in estados['data'].unique():
    if pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png' not in prontas:

      mu = municipios[municipios.data==d]
      if len(mu)>0:
        fig, ax = plt.subplots(nrows=1, ncols=3,figsize=(9, 2.96), dpi=300)

        municipalities = geobr.read_municipality(code_muni='all')
        municipalities['code_muni']=municipalities['code_muni'].apply(corta)

        municipalities = municipalities.merge(mu,how='left',left_on='code_muni',right_on='codmun')
        municipalities['cor']=municipalities.crescimentoobito.apply(cor)

        municipalities['crescimentocasos'].fillna(1)
        municipalities.plot(column='crescimentocasos',cmap='viridis', ax=ax[1],vmax=5)
        ax[1].axis('off')

        plotting(municipalities,1,'#577590',ax[2])
        plotting(municipalities,2,'#F9C74F',ax[2])
        plotting(municipalities,3,'#F8961E',ax[2])
        plotting(municipalities,4,'#F94144',ax[2])
        ax[2].axis('off')
        fig.suptitle(pd.to_datetime(str(d)).strftime("%d/%m"), fontsize=10)

        locais=geobr.read_municipal_seat()
        states=geobr.read_state(code_state='all')
        locais['code_muni']=locais['code_muni'].apply(corta)

        locais = locais.merge(mu,how='left',left_on='code_muni',right_on='codmun')
        locais['cor']=locais.crescimentoobito.apply(cor)
        states.plot(ax=ax[0],edgecolor="#577590",facecolor="white",linewidth=0.4)

        locais.plot(column='crescimentocasos',cmap='viridis', ax=ax[0], markersize='crescimentocasos', vmax=5)
        ax[0].axis('off')

        fig.savefig(f'/content/drive/My Drive/covid/img/nacional_mun/'+pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png')
        plt.close()


def plota_nacional_state():

  prontas = listdir('/content/drive/My Drive/covid/img/nacional_mun/')

  for d in estados['data'].unique():
    if pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png' not in prontas:

      es = estados[estados.data==d]
      if len(es)>0:
        fig, ax = plt.subplots(nrows=1, ncols=2,figsize=(6, 2.96), dpi=300)

        states = geobr.read_state(code_state='all')
        states = states.merge(es,how='left',left_on='code_state',right_on='coduf')
        states['cor']=states.crescimentoobito.apply(cor)

        states.plot(column='crescimentocasos',cmap='viridis', ax=ax[0],vmax=5)
        ax[0].axis('off')
        fig.suptitle(pd.to_datetime(str(d)).strftime("%d/%m"), fontsize=10)


        plotting(states,1,'#577590',ax[1])
        plotting(states,2,'#F9C74F',ax[1])
        plotting(states,3,'#F8961E',ax[1])
        plotting(states,4,'#F94144',ax[1])
        ax[1].axis('off')

        fig.savefig(f'/content/drive/My Drive/covid/img/nacional/'+pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png')
        plt.close()

def plota_estadual():

  for es in estados['estado'].unique():

    mu = municipios[municipios.estado==es]

    prontas = listdir('/content/drive/My Drive/covid/img/estadual/'+str(es)+'/')

    for d in estados['data'].unique():
      if pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png' not in prontas:

        mu = mu[mu.data==d]
        if len(mu)>0:

          fig, ax = plt.subplots(nrows=1, ncols=2,figsize=(6, 2.96), dpi=300)

          municipalities = geobr.read_municipality(code_muni=es)
          municipalities['code_muni']=municipalities['code_muni'].apply(corta)

          municipalities = municipalities.merge(mu,how='left',left_on='code_muni',right_on='codmun')
          municipalities['cor']=municipalities.crescimentoobito.apply(cor)

          municipalities['crescimentocasos'].fillna(1)
          municipalities.plot(column='crescimentocasos',cmap='viridis', ax=ax[1],vmax=5)
          ax[1].axis('off')

          plotting(municipalities,1,'#577590',ax[2])
          plotting(municipalities,2,'#F9C74F',ax[2])
          plotting(municipalities,3,'#F8961E',ax[2])
          plotting(municipalities,4,'#F94144',ax[2])
          ax[2].axis('off')
          fig.suptitle(pd.to_datetime(str(d)).strftime("%d/%m"), fontsize=10)

          locais=geobr.read_municipal_seat()
          states=geobr.read_state(code_state=es)
          locais['code_muni']=locais['code_muni'].apply(corta)

          locais = locais.merge(mu,how='left',left_on='code_muni',right_on='codmun')
          locais['cor']=locais.crescimentoobito.apply(cor)
          states.plot(ax=ax[0],edgecolor="#577590",facecolor="white",linewidth=0.4)

          locais.plot(column='crescimentocasos',cmap='viridis', ax=ax[0], markersize='crescimentocasos', vmax=5)
          ax[0].axis('off')

          fig.savefig(f'/content/drive/My Drive/covid/img/estadual/'+str(es)+'/'+pd.to_datetime(str(d)).strftime("%m %d %Y")+'.png')
          plt.close()
  

In [18]:
plota_nacional_mun()
criagif('nacional_mun')
plota_nacional_state()
criagif('nacional')
plota_estadual()
for es in estados['estado'].unique():
  criagif('estadual/'+str(es))

from __future__ import print_function  # for Python2
import sys

local_vars = list(locals().items())
for var, obj in local_vars:
    print(var, sys.getsizeof(obj))

FileNotFoundError: ignored

In [14]:
estados

,estado,municipio,coduf,codmun,data,casosAcumulado,casosNovos,obitosAcumulado,obitosNovos,mediamovelcaso,mediamovelobito,crescimentoobito,crescimentocasos
882,AP,NaN,16,NaN,2020-02-25,0,0,0,0,NaN,NaN,NaN,0.000000
883,AP,NaN,16,NaN,2020-02-26,0,0,0,0,NaN,NaN,NaN,0.000000
884,AP,NaN,16,NaN,2020-02-27,0,0,0,0,NaN,NaN,NaN,0.000000
885,AP,NaN,16,NaN,2020-02-28,0,0,0,0,NaN,NaN,NaN,0.000000
886,AP,NaN,16,NaN,2020-02-29,0,0,0,0,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171,TO,NaN,17,NaN,2020-07-16,16672,641,278,7,403.857143,5.428571,0.461538,4.221988
1172,TO,NaN,17,NaN,2020-07-17,16954,282,283,5,349.285714,5.428571,0.520000,4.229272
1173,TO,NaN,17,NaN,2020-07-18,17209,255,288,5,324.285714,5.285714,0.541667,4.235756
1174,TO,NaN,17,NaN,2020-07-19,17773,564,294,6,377.285714,5.571429,0.392857,4.249761
